# Research design and leakage-safe protocol

This notebook operationalizes `S.md` as an ERA5 proof-of-concept. The claims are intentionally bounded to the executed experiment: persistent event-aware memory, not a completed multi-domain foundation model.

In [1]:
from pathlib import Path
import json, sys
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from pmwm.common import load_config
config = load_config(ROOT / 'config.yaml')
print(f"Project root: {ROOT}")

Project root: /home/urad/S


In [2]:
import yaml
print(yaml.safe_dump(config, sort_keys=False)[:8000])

project:
  name: Persistent Memory World Model for ERA5
  acronym: PMWM
  seed: 3407
data:
  source_url: gs://weatherbench2/datasets/era5/1959-2023_01_10-6h-64x32_equiangular_conservative.zarr
  source_name: WeatherBench2 ERA5, 6-hourly, 64x32 conservative grid
  start_year: 1959
  end_year: 2022
  stream_block_years: 8
  dask_workers: 16
  variables:
  - 2m_temperature
  - mean_sea_level_pressure
  - 10m_u_component_of_wind
  - 10m_v_component_of_wind
  - total_precipitation_6hr
  sites:
  - name: Seoul
    lat: 37.57
    lon: 126.98
    zone: continental
    holdout: false
  - name: New York
    lat: 40.71
    lon: -74.01
    zone: temperate
    holdout: false
  - name: London
    lat: 51.51
    lon: -0.13
    zone: temperate
    holdout: false
  - name: Sydney
    lat: -33.87
    lon: 151.21
    zone: temperate
    holdout: false
  - name: Singapore
    lat: 1.35
    lon: 103.82
    zone: tropical
    holdout: true
  - name: Sao Paulo
    lat: -23.55
    lon: -46.63
    zone: tropic

## Protocol

- ERA5 is accessed lazily from the public WeatherBench2 Zarr store.
- 1959–1994 trains the backbone and memory; 1995–2004 is validation-only; 2005–2022 is untouched test.
- Singapore, Dubai, Paris, and Moscow are excluded from model and memory training for spatial transfer evaluation.
- All retrieval memories carry training-era origins, and the test stream is evaluated prequentially before optional adaptation.